# 03 – Train U-Net: Crack Detection

**CSE445 – Road Damage Detection & Lane Segmentation**

This notebook:
1. Loads train / val split manifests from `data/crack/splits/`
2. Builds `SegmentationDataset` + `DataLoader` with augmentation
3. Instantiates U-Net + Adam + `BCEDiceLoss`
4. Trains for up to 50 epochs with early stopping
5. Plots learning curves (train vs val loss & IoU)

**Prerequisites**: Run `01_EDA_crack.ipynb` first (generates split CSVs).

In [ ]:
import sys, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/Road_Damage_Project'
    os.environ['RUN_ENV'] = 'colab'
except ImportError:
    REPO = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
    os.environ['RUN_ENV'] = 'local'

if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('Repo root:', REPO)

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

import config
from src.shared.dataset   import SegmentationDataset
from src.shared.transforms import get_train_transforms, get_val_transforms
from src.shared.unet      import UNet
from src.shared.losses    import BCEDiceLoss
from src.shared.trainer   import Trainer

print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))

## 1. Dataset & DataLoaders

In [ ]:
cfg = config.CRACK_UNET
img_size = (config.IMG_HEIGHT, config.IMG_WIDTH)

train_ds = SegmentationDataset(
    config.CRACK_SPLIT_DIR / 'train.csv',
    transform=get_train_transforms(img_size)
)
val_ds = SegmentationDataset(
    config.CRACK_SPLIT_DIR / 'val.csv',
    transform=get_val_transforms(img_size)
)

train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} samples, {len(train_loader)} batches')
print(f'Val  : {len(val_ds)}   samples, {len(val_loader)}   batches')

## 2. Model, Loss, Trainer

In [ ]:
model = UNet(
    in_channels   = cfg['in_channels'],
    out_channels  = cfg['out_channels'],
    base_features = cfg['base_features']
)
print(f'U-Net parameters: {model.count_parameters():,}')

loss_fn = BCEDiceLoss(bce_weight=cfg['bce_weight'], dice_weight=cfg['dice_weight'])
print('Loss function:', loss_fn)

trainer = Trainer(model, loss_fn, cfg, train_loader, val_loader)
print('Trainer ready. Experiment dir:', trainer.exp_dir)

## 3. Train

In [ ]:
# To run a quick sanity-check, set n_epochs=3
# For full training, leave n_epochs=None (uses cfg['epochs'] = 50)
history = trainer.fit(n_epochs=None)

## 4. Learning Curves (Overfitting Analysis)

In [ ]:
log_path = trainer.log_csv_path
df = pd.read_csv(log_path)
print(df.tail(10).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('U-Net Crack Detection – Learning Curves', fontsize=13, fontweight='bold')

# Loss curves
axes[0].plot(df['epoch'], df['train_loss'], label='Train Loss', color='#1565C0', linewidth=2)
axes[0].plot(df['epoch'], df['val_loss'],   label='Val Loss',   color='#E53935', linewidth=2, linestyle='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCEDice Loss')
axes[0].set_title('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# IoU curves
axes[1].plot(df['epoch'], df['train_iou'], label='Train IoU', color='#1565C0', linewidth=2)
axes[1].plot(df['epoch'], df['val_iou'],   label='Val IoU',   color='#E53935', linewidth=2, linestyle='--')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('IoU')
axes[1].set_title('IoU (Intersection over Union)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
curves_path = trainer.exp_dir / 'curves.png'
plt.savefig(curves_path, dpi=120, bbox_inches='tight')
plt.show()
print('Saved:', curves_path)

## 5. Overfitting Analysis

The plot above shows the train vs. validation curves. Key observations to record:

- If **train loss << val loss** and the gap widens over epochs → **overfitting**
  - Mitigation: increase augmentation, add dropout, reduce model size, use weight decay
- If both curves plateau early → **underfitting**
  - Mitigation: increase model capacity (base_features=64), lower LR, more epochs
- Early stopping checkpoint saved at epoch with best val IoU

**Next**: `04_train_lane_unet.ipynb`